# Decision Tree Classifier — UCI Iris Dataset

This notebook downloads the **Iris** dataset directly from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/53/iris), trains a Decision Tree classifier, and evaluates it with a held-out test set and stratified 5-fold cross-validation.

**Included evaluation:** accuracy, balanced accuracy, macro/weighted precision, recall and F1, multiclass ROC-AUC, log loss, multiclass Brier score, Matthews correlation coefficient, Cohen’s kappa, per-class specificity, classification report, confusion matrix, and feature importance.

Run the cells **from top to bottom**. An internet connection is required to download the dataset.

## 1. Install dependencies

Run this cell if the required packages are not already installed in your environment. In Jupyter, `%pip` installs packages into the active kernel.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib

## 2. Import libraries and set configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/53/data.csv"
RANDOM_STATE = 42
TARGET = "class"

## 3. Download and prepare the dataset

The target column is `class`; the remaining columns are used as numerical input features. Any rows with missing target or feature values are excluded.

In [ ]:
# TODO
df = pd.read_csv(...)

In [ ]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

if TARGET not in df.columns:
    raise ValueError(
        f"Expected target column '{TARGET}'. Found columns: {df.columns.tolist()}"
    )

In [ ]:
# TODO
df = df.dropna(subset=[TARGET]).copy()
y = ...
X = ...

In [ ]:
valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

In [ ]:
print(f"Dataset URL: {DATA_URL}")
print(f"Number of rows: {len(X)}")
print(f"Number of features: {X.shape[1]}")
print("\nClass distribution:")
display(y.value_counts().sort_index().rename("count").to_frame())

display(X.head())

## 4. Split into training and test sets

A stratified 80/20 split preserves approximately the same class distribution in both sets. The test set is held out until final evaluation.

In [ ]:
# TODO
X_train, X_test, y_train, y_test = train_test_split(...)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## 5. Define the Decision Tree and run 5-fold cross-validation

The model uses the Gini impurity criterion. Feature scaling is not required for a Decision Tree. Cross-validation is performed only on the training split to avoid leaking test information.

In [ ]:
# TODO
model = DecisionTreeClassifier(...)

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
}

In [ ]:
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1,
)

In [ ]:
cv_summary = pd.DataFrame(
    {
        "mean": [cv_results[f"test_{name}"].mean() for name in scoring],
        "std_dev": [cv_results[f"test_{name}"].std(ddof=1) for name in scoring],
    },
    index=list(scoring.keys()),
)

display(cv_summary.style.format("{:.4f}"))

## 6. Train the final model and calculate overall test metrics

For most metrics, higher values are better. **Log loss** and **Brier score** are error measures, so lower values are better.

In [ ]:
# TODO
model.fit(...)

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
classes = model.classes_

In [ ]:
# Multiclass Brier score: mean sum of squared probability errors per sample.
y_onehot = (
    pd.get_dummies(y_test)
    .reindex(columns=classes, fill_value=0)
    .to_numpy()
)
brier_score = np.mean(np.sum((y_proba - y_onehot) ** 2, axis=1))

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Recall / sensitivity (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "F1-score (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "Precision (weighted)": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "Recall (weighted)": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "F1-score (weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "ROC-AUC (macro, one-vs-rest)": roc_auc_score(
        y_test, y_proba, labels=classes, multi_class="ovr", average="macro"
    ),
    "Log loss": log_loss(y_test, y_proba, labels=classes),
    "Multiclass Brier score": brier_score,
    "Matthews correlation coefficient": matthews_corrcoef(y_test, y_pred),
    "Cohen's kappa": cohen_kappa_score(y_test, y_pred),
}

metrics_df = pd.DataFrame(
    {"metric": list(metrics.keys()), "value": list(metrics.values())}
)
display(metrics_df.style.format({"value": "{:.4f}"}))

print(f"Tree depth: {model.get_depth()}")
print(f"Number of leaves: {model.get_n_leaves()}")

## 7. Classification report, per-class specificity, and confusion matrix

Specificity is calculated one-vs-rest for each class: **TN / (TN + FP)**. In the confusion matrix, rows represent actual classes and columns represent predicted classes.

In [ ]:
print("Classification report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=classes,
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(y_test, y_pred, labels=classes)

tp = np.diag(cm).astype(float)
fn = cm.sum(axis=1) - tp
fp = cm.sum(axis=0) - tp
tn = cm.sum() - tp - fn - fp
specificity = np.divide(
    tn,
    tn + fp,
    out=np.zeros_like(tn),
    where=(tn + fp) != 0,
)

per_class_metrics = pd.DataFrame(
    {
        "class": classes,
        "precision": precision_score(
            y_test, y_pred, labels=classes, average=None, zero_division=0
        ),
        "recall_sensitivity": recall_score(
            y_test, y_pred, labels=classes, average=None, zero_division=0
        ),
        "specificity": specificity,
        "f1_score": f1_score(
            y_test, y_pred, labels=classes, average=None, zero_division=0
        ),
        "support": cm.sum(axis=1).astype(int),
    }
)

display(per_class_metrics.style.format({
    "precision": "{:.4f}",
    "recall_sensitivity": "{:.4f}",
    "specificity": "{:.4f}",
    "f1_score": "{:.4f}",
}))

confusion_df = pd.DataFrame(cm, index=classes, columns=classes)
confusion_df.index.name = "Actual / Predicted"
display(confusion_df)

## 8. Feature importance

Feature importance reflects each feature's contribution to reducing impurity in this fitted tree. It is useful for interpretation, but should not be treated as proof of causation.

In [ ]:
feature_importance = pd.Series(
    model.feature_importances_,
    index=X.columns,
    name="importance",
).sort_values(ascending=False)

display(feature_importance.to_frame().style.format("{:.4f}"))

ax = feature_importance.sort_values().plot(kind="barh", figsize=(8, 4))
ax.set_title("Decision Tree Feature Importance")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

## 9. Visualize the trained decision tree and confusion matrix

In [ ]:
plt.figure(figsize=(16, 9))
plot_tree(
    model,
    feature_names=X.columns.tolist(),
    class_names=classes.tolist(),
    filled=True,
    rounded=True,
    fontsize=9,
)
plt.title("Decision Tree Classifier — UCI Iris")
plt.tight_layout()
plt.show()

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes,
).plot(values_format="d", xticks_rotation=35)
plt.title("Confusion Matrix — Held-out Test Set")
plt.tight_layout()
plt.show()

## Metric interpretation

- **Accuracy:** fraction of all predictions that are correct.
- **Precision:** of the samples predicted as a class, the fraction that truly belong to it.
- **Recall / sensitivity:** of the samples belonging to a class, the fraction correctly identified.
- **Specificity:** of the samples outside a class, the fraction correctly rejected as that class.
- **F1-score:** harmonic mean of precision and recall.
- **Balanced accuracy:** mean recall across classes; less affected by class imbalance than raw accuracy.
- **ROC-AUC:** ability to rank a true class above other classes using predicted probabilities; reported here as macro one-vs-rest.
- **Log loss / Brier score:** probability prediction errors; lower values are better.
- **Matthews correlation coefficient / Cohen’s kappa:** summary measures that account for both correct and incorrect predictions.
- **Cross-validation mean ± standard deviation:** average training-fold performance and variation across folds.